In [ ]:
import os
import pathlib
import pandas as pd

In [ ]:
root_path = pathlib.Path(os.getcwd())
root_path = root_path.parents[1]

In [ ]:
dataframe_product_detail = pd.read_excel(
    os.path.join(
        root_path,
        "data_folder",
        "product_detail_export.xlsx",
    )
)
dataframe_product_detail

In [ ]:
dataframe_product_detail.dtypes

In [ ]:
dataframe_product_detail = dataframe_product_detail.rename(
    columns={
        "DISTRIBUTEUR": "distributor",
        "SOURCE DONNEES": "data_source",
        "CODE PRODUIT": "product_code",
        "DESCRIPTION": "description",
        "MARQUE": "brand",
        "INDUSTRIEL": "industrial",
        "UF": "unit",
        "Qté Facture": "quantity",
        "Montant HT": "amount_ht"
    }
)
dataframe_product_detail

In [ ]:
dataframe_product_detail_distributor = dataframe_product_detail["distributor"].isna().sum()
dataframe_product_detail_distributor

In [ ]:
dataframe_product_detail.dropna(how="all", inplace=True)
dataframe_product_detail

In [ ]:
dataframe_product_detail.drop(index=719, inplace=True)
dataframe_product_detail

In [ ]:
dataframe_product_detail_filtered = dataframe_product_detail[~dataframe_product_detail["data_source"].str.contains("DECLARATIF DISTRIBUTEUR")]
dataframe_product_detail_filtered["data_source"] = dataframe_product_detail_filtered["data_source"].str.replace("AUTRES - ", "")
dataframe_product_detail_filtered

In [ ]:
dataframe_product_detail_declaratif_distributeur = dataframe_product_detail[dataframe_product_detail["data_source"].str.contains("DECLARATIF DISTRIBUTEUR")]
dataframe_product_detail_declaratif_distributeur

In [ ]:
dataframe_product_detail_declaratif_distributeur["data_source"] = dataframe_product_detail_declaratif_distributeur["data_source"].str.replace("DECLARATIF DISTRIBUTEUR", "")
dataframe_product_detail_declaratif_distributeur

In [ ]:
dataframe_product_detail_declaratif_distributeur["product_code"] = dataframe_product_detail_declaratif_distributeur.apply(
    lambda row: "to_be_completed" if pd.isnull(row["brand"]) or row["brand"] == "" or row["brand"] == "." else row["brand"],
    axis=1
)

dataframe_product_detail_declaratif_distributeur

In [ ]:
dataframe_product_detail_declaratif_distributeur["brand"] = ""
dataframe_product_detail_declaratif_distributeur

In [ ]:
dataframe_product_detail_final = pd.concat([dataframe_product_detail_filtered, dataframe_product_detail_declaratif_distributeur], ignore_index=True)
dataframe_product_detail_final

In [ ]:
del dataframe_product_detail_filtered
del dataframe_product_detail_declaratif_distributeur
del dataframe_product_detail_distributor
del dataframe_product_detail

In [ ]:
dataframe_product_detail_final["amount_ht"] = dataframe_product_detail_final["amount_ht"].round(2)
dataframe_product_detail_final

In [ ]:
dataframe_product_detail_final["unit"] = dataframe_product_detail_final["unit"].str.upper()

dictionary_unit = {
    # BOCAL
    "BCL": "BOCAL",
    # BIDON
    "BID": "BIDON",
    # BOUTEILLE
    "BLLE": "BOUTEILLE",
    # BOÎTE
    "BT": "BOÎTE",
    "BT.": "BOÎTE",
    "BTE": "BOÎTE",
    "BOITE": "BOÎTE",
    # BRIQUE
    "BRQ": "BRIQUE",
    # COFFRET
    "CO": "COFFRET",
    "COF": "COFFRET",
    # COLIS
    "COL": "COLIS",
    # FLACON
    "FLC": "FLACON",
    # PIÈCE
    "PI": "PIÈCE",
    # POCHE
    "PCH": "POCHE",
    # SEAU
    "SEA": "SEAU",
    # UNITÉ
    "U": "UNITÉ"
}

dataframe_product_detail_final["unit"] = dataframe_product_detail_final["unit"].replace(dictionary_unit)

In [ ]:
dataframe_product_detail_final["product_code"].unique()

In [ ]:
code_to_brand = {
    # AMORA
    "AMORASQUEEZE": "AMORA",
    "AMORADOSETTE": "AMORA",
    "AMORASAUCE5L": "AMORA",
    "AMORASAUCE1L": "AMORA",
    "AMORASEAU": "AMORA",
    "SAVORA": "AMORA",
    # HELLEMANS
    "HELLEMANSQUEEZE": "HELLMANN'S",
    # KNORR
    "KNORRBOUILLON": "KNORR",
    "KNORRFOND": "KNORR",
    "KNORRFUMET": "KNORR",
    "KNORRJUS": "KNORR",
    "KNORRLIQUIDE": "KNORR",
    "KNORRMEP": "KNORR",
    "KNORRPESTO": "KNORR",
    "KNORRROUX": "KNORR",
    "KNORRSAUCE": "KNORR",
    "KNORR 1 2 3": "KNORR",
    "KNORR 123": "KNORR",
    "KNORR ESSENTIEL": "KNORR",
    "VIANDOX": "KNORR",
    # MAILLE
    "MAILLEVINAIGRETTE": "MAILLE",
    # MAIZENA
    "MAIZENAPETIT": "MAIZENA",
    "MAIZENAGRAND": "MAIZENA",
    # TABASCO
    "TABASCO350ML": "TABASCO",
    "TABASCO60ML": "TABASCO",
    "TABASCO150ML": "TABASCO"
}

dataframe_product_detail_final["product_code"] = dataframe_product_detail_final["product_code"].replace(code_to_brand)
dataframe_product_detail_final

In [ ]:
dataframe_product_detail_final["product_code"].unique()

In [ ]:
description_contains_brand = ["AMORA", "HELLMANN'S", "KNORR", "MAILLE", "MAIZENA", "TABASCO", "LIPTON", "ELEPHANT"]

for brand in description_contains_brand:
    dataframe_product_detail_final.loc[
        (dataframe_product_detail_final["product_code"] == "to_be_completed") &
        (dataframe_product_detail_final["description"].str.contains(brand, na=False)),
        "product_code"
    ] = brand
dataframe_product_detail_final

In [ ]:
for column in dataframe_product_detail_final.select_dtypes(include=["object"]):
    dataframe_product_detail_final[column] = dataframe_product_detail_final[column].str.title().str.replace(r"(?<=')([A-Z])", lambda value: value.group(0).lower(), regex=True)

In [ ]:
dataframe_product_detail_final.to_excel(
    os.path.join(
        root_path,
        "data_folder",
        "product_detail_export_cleaned.xlsx",
    ),
    index=False
)